<font color='red' size='6'><b>01. 데이터셋 구축 (하나의 DataFrame으로 통합)</b></font>

- 목적 : 상권분석 데이터 12개 파일을 **상권 × 업종 × 분기** 한 행으로 합친 데이터셋(`dataset`) 만들기
- 기준 : 추정매출 2023~2025 (12개 분기, 62개 업종)
- 결과 : `00_data/processed/dataset_상권_업종_분기.csv`

| 번호 | 파일 | 단위 | 기간 |
|:--|:--|:--|:--|
| 1 | 점포-상권 | 상권×업종×분기 | 2025 |
| 2 | 상권변화지표-상권 | 상권×분기 | 2021~2026 |
| 3 | 추정매출-상권 (2023·2024·2025) | 상권×업종×분기 | 2023~2025 |
| 4 | 길단위인구-상권 | 상권×분기 | 2021~2026 |
| 5 | 상주인구-상권 | 상권×분기 | 2021~2026 |
| 6 | 상주인구-상권배후지 | 배후지(골목상권)×분기 | 2021~2026 |
| 7 | 상주인구-서울시 | 서울×분기 | 2021~2026 |
| 8 | 상주인구-자치구 | 자치구×분기 | 2021~2026 |
| 9 | 상주인구-행정동 | 행정동×분기 | 2021~2026 |
| 10 | 영역-상권 | 상권 | - |

# 1. 패키지로드 & 한글설정 & 경고메세지 ignore

In [1]:
# 패키지 import
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# 시각화의 선명도를 높임
%config InlineBackend.figure_format = "retina"

# 한글설정 및 미적속성 설정
sns.set(style='white',
        context='notebook',
        palette='Dark2',
        rc={'figure.figsize': (12,3),
            'font.family':'Malgun Gothic',
            'axes.unicode_minus':False})

# warning(경고) 안보이게
import warnings
warnings.filterwarnings(action='ignore')

# 모든 컬럼이 보이도록
pd.set_option('display.max_columns', None)

# 2. 데이터 파일 확인
- 아래 `%ls` 결과와 **2.1의 파일 이름이 다르면 2.1만 수정**한다

In [2]:
%ls "00_data/"

 C 드라이브의 볼륨에는 이름이 없습니다.
 볼륨 일련 번호: A014-DCD6

 C:\ai\lecNote\09_project\00_data 디렉터리

2026-09-29  오후 12:29    <DIR>          .
2026-09-29  오후 12:56    <DIR>          ..
2026-09-29  오후 12:27        31,051,537 1_서울시 상권분석서비스(점포-상권)_2025년.csv
2026-09-29  오후 12:27         3,124,101 2_서울시 상권분석서비스(상권변화지표-상권).csv
2026-09-29  오후 12:27        40,887,998 3_2023년 서울시_상권분석서비스(추정매출-상권).csv
2026-09-29  오후 12:27        40,356,222 3_2024년 서울시 상권분석서비스(추정매출-상권).csv
2026-09-29  오후 12:27        39,698,674 3_2025년 서울시 상권분석서비스(추정매출-상권).csv
2026-09-29  오후 12:27         8,543,807 4_ 서울시 상권분석서비스(길단위인구-상권).csv
2026-09-29  오후 12:27         6,687,358 5_ 서울시 상권분석서비스(상주인구-상권).csv
2026-09-29  오후 12:27         4,994,974 6_서울시 상권분석서비스(상주인구-상권배후지).csv
2026-09-29  오후 12:27             6,061 7_서울시 상권분석서비스(상주인구-서울시).csv
2026-09-29  오후 12:27           120,545 8_서울시 상권분석서비스(상주인구-자치구).csv
2026-09-29  오후 12:27         1,818,472 9_서울시 상권분석서비스(상주인구-행정동).csv
2026-09-28  오후 03:08    <DIR>          processed
2026-09-28  오후 05:34   

## 2.1 파일 경로

In [ ]:
DATA_DIR = '00_data/'

file_store    = DATA_DIR + '1_2025년_서울시_상권분석서비스_점포-상권_.csv'
file_change   = DATA_DIR + '2_서울시_상권분석서비스_상권변화지표-상권_.csv'
file_sales    = [DATA_DIR + '3_2023년_서울시_상권분석서비스_추정매출-상권_.csv',
                 DATA_DIR + '3_2024년_서울시_상권분석서비스_추정매출-상권_.csv',
                 DATA_DIR + '3_2025년_서울시_상권분석서비스_추정매출-상권_.csv']
file_flow     = DATA_DIR + '4_서울시_상권분석서비스_길단위인구-상권_.csv'
file_res      = DATA_DIR + '5_서울시_상권분석서비스_상주인구-상권_.csv'
file_res_bg   = DATA_DIR + '6_서울시_상권분석서비스_상주인구-상권배후지_.csv'
file_res_seoul= DATA_DIR + '7_서울시_상권분석서비스_상주인구-서울시_.csv'
file_res_gu   = DATA_DIR + '8_서울시_상권분석서비스_상주인구-자치구_.csv'
file_res_dong = DATA_DIR + '9_서울시_상권분석서비스_상주인구-행정동_.csv'
file_area     = DATA_DIR + '10_서울시_상권분석서비스_영역-상권_.csv'

# 3. 데이터 읽어오기
- 모든 파일은 cp949 인코딩
## 3.1 추정매출 2023~2025 : 연도별 파일을 행으로 연결(concat)

In [ ]:
sales_list = []
for file in file_sales:
    temp = pd.read_csv(file, encoding='cp949', low_memory=False)
    print(file.split('/')[-1], temp.shape)
    sales_list.append(temp)

sales = pd.concat(sales_list, axis=0, ignore_index=True)   # axis=0 : 행으로 연결
print('연결 후 :', sales.shape)
sales['기준_년분기_코드'].value_counts().sort_index()

## 3.2 나머지 파일

In [ ]:
store     = pd.read_csv(file_store, encoding='cp949', low_memory=False)
change    = pd.read_csv(file_change, encoding='cp949')
flow      = pd.read_csv(file_flow, encoding='cp949')
res       = pd.read_csv(file_res, encoding='cp949')
res_bg    = pd.read_csv(file_res_bg, encoding='cp949')
res_seoul = pd.read_csv(file_res_seoul, encoding='cp949')
res_gu    = pd.read_csv(file_res_gu, encoding='cp949')
res_dong  = pd.read_csv(file_res_dong, encoding='cp949')
area      = pd.read_csv(file_area, encoding='cp949')

for name, d in [('점포',store), ('상권변화지표',change), ('유동인구',flow), ('상주-상권',res), ('상주-배후지',res_bg),
                ('상주-서울시',res_seoul), ('상주-자치구',res_gu), ('상주-행정동',res_dong), ('영역-상권',area)]:
    print(f'{name:8s}', d.shape)

# 4. 컬럼명 통일
## 4.1 점포 : 영문 → 한글

In [ ]:
store = store.rename(columns={'stdr_yyqu_cd':'기준_년분기_코드',
                              'trdar_se_cd':'상권_구분_코드',
                              'trdar_se_cd_nm':'상권_구분_코드_명',
                              'trdar_cd':'상권_코드',
                              'trdar_cd_nm':'상권_코드_명',
                              'svc_induty_cd':'서비스_업종_코드',
                              'svc_induty_cd_nm':'서비스_업종_코드_명',
                              'stor_co':'점포_수',                        # 일반 점포(프랜차이즈 제외)
                              'similr_induty_stor_co':'유사_업종_점포_수',  # = 점포_수 + 프랜차이즈_점포_수
                              'opbiz_rt':'개업_율',
                              'opbiz_stor_co':'개업_점포_수',
                              'clsbiz_rt':'폐업_률',
                              'clsbiz_stor_co':'폐업_점포_수',
                              'frc_stor_co':'프랜차이즈_점포_수'})
# 전체 점포 수 (폐업률의 분모)
store['전체_점포_수'] = store['점포_수'] + store['프랜차이즈_점포_수']
store.head(3)

## 4.2 추정매출 : 시간대 컬럼명 정리
- `시간대_00~06_매출_금액` → `시간대_00_06_매출_금액`
- 깨진 이름 `시간대_건수~06_매출_건수` → `시간대_00_06_매출_건수`

In [ ]:
time_cols = {}
for start, end in [('00','06'),('06','11'),('11','14'),('14','17'),('17','21'),('21','24')]:
    time_cols[f'시간대_{start}~{end}_매출_금액'] = f'시간대_{start}_{end}_매출_금액'
    time_cols[f'시간대_건수~{end}_매출_건수']    = f'시간대_{start}_{end}_매출_건수'
sales = sales.rename(columns=time_cols)
[col for col in sales.columns if '시간대' in col]

# 5. 상권 매핑표 (상권 → 자치구·행정동·위경도)
- 좌표 EPSG:5181 → 위경도 EPSG:4326 변환 (지도용)

In [ ]:
from pyproj import Transformer
tf = Transformer.from_crs('EPSG:5181', 'EPSG:4326', always_xy=True)
area['경도'], area['위도'] = tf.transform(area['엑스좌표_값'].values, area['와이좌표_값'].values)

mapping = area[['상권_코드','자치구_코드','자치구_코드_명','행정동_코드','행정동_코드_명','위도','경도','영역_면적']]
mapping.head(3)

# 6. 병합 → 하나의 DataFrame (`dataset`)
- 기준 : 추정매출 2023~2025 (`how='left'` : 기준 행은 모두 남기고 옆에 붙임)
- 붙이는 쪽의 이름 컬럼(상권_코드_명 등)은 빼서 `_x`, `_y` 중복을 막는다

In [ ]:
key3 = ['기준_년분기_코드','상권_코드','서비스_업종_코드']   # 상권×업종×분기
key2 = ['기준_년분기_코드','상권_코드']                     # 상권×분기
drop_names = ['상권_구분_코드','상권_구분_코드_명','상권_코드_명','서비스_업종_코드_명']

dataset = sales.copy()
print('기준(추정매출) :', dataset.shape)

## 6.1 + 점포 (2025년만 있음 → 2023·2024년 행은 NaN)

In [ ]:
dataset = dataset.merge(store.drop(columns=drop_names), on=key3, how='left')
dataset['점포_데이터_있음'] = dataset['전체_점포_수'].notna().astype(int)
print(dataset.shape)
dataset.groupby('기준_년분기_코드')['점포_데이터_있음'].mean()

## 6.2 + 유동인구, 상주인구, 상권변화지표 (상권 × 분기)

In [ ]:
dataset = dataset.merge(flow.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
dataset = dataset.merge(res.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
dataset = dataset.merge(change.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
dataset.shape

## 6.3 + 배후지 상주인구 (골목상권만 제공, `상권배후지_코드` = `상권_코드`)

In [ ]:
res_bg2 = res_bg.rename(columns={'상권배후지_코드':'상권_코드',
                                 '총_상주인구_수':'배후지_총_상주인구_수',
                                 '총_가구_수':'배후지_총_가구_수'})
res_bg2 = res_bg2[['기준_년분기_코드','상권_코드','배후지_총_상주인구_수','배후지_총_가구_수']]
dataset = dataset.merge(res_bg2, on=key2, how='left')
dataset.shape

## 6.4 + 상권 매핑 (자치구·행정동·위경도)

In [ ]:
dataset = dataset.merge(mapping, on='상권_코드', how='left')
dataset.shape

## 6.5 + 자치구·행정동·서울시 상주인구

In [ ]:
gu = res_gu[['기준_년분기_코드','자치구_코드','총_상주인구_수','총_가구_수']]
gu = gu.rename(columns={'총_상주인구_수':'자치구_총_상주인구_수', '총_가구_수':'자치구_총_가구_수'})

dong = res_dong[['기준_년분기_코드','행정동_코드','총_상주인구_수','총_가구_수']]
dong = dong.rename(columns={'총_상주인구_수':'행정동_총_상주인구_수', '총_가구_수':'행정동_총_가구_수'})

seoul = res_seoul[['기준_년분기_코드','총_상주인구_수']]
seoul = seoul.rename(columns={'총_상주인구_수':'서울_총_상주인구_수'})

dataset = dataset.merge(gu, on=['기준_년분기_코드','자치구_코드'], how='left')
dataset = dataset.merge(dong, on=['기준_년분기_코드','행정동_코드'], how='left')
dataset = dataset.merge(seoul, on='기준_년분기_코드', how='left')
dataset.shape

## 6.6 사용하지 않을 컬럼 제거
- `아파트_가구_수` : 모든 행이 0 / `비_아파트_가구_수` : 총_가구_수와 같음 / `상권_구분_코드` : 한글명과 중복

In [ ]:
dataset = dataset.drop(columns=['아파트_가구_수','비_아파트_가구_수','상권_구분_코드'])
dataset = dataset.sort_values(['상권_코드','서비스_업종_코드','기준_년분기_코드']).reset_index(drop=True)
dataset.shape

# 7. 검증
- 행 수가 추정매출과 같아야 함 / 상권×업종×분기 키 중복 0 / `_x`, `_y` 컬럼 없음

In [ ]:
print('추정매출 행 수 :', len(sales), '/ dataset 행 수 :', len(dataset))
print('키 중복 :', dataset.duplicated(key3).sum())
print('_x, _y 컬럼 :', [col for col in dataset.columns if col.endswith(('_x','_y'))])
print('상권 수 :', dataset['상권_코드'].nunique(), '/ 업종 수 :', dataset['서비스_업종_코드'].nunique(),
      '/ 자치구 수 :', dataset['자치구_코드_명'].nunique())

In [ ]:
# 결합률 : 붙인 데이터가 몇 % 행에 채워졌는지
check_cols = {'점포(2025만)':'전체_점포_수',
              '유동인구':'총_유동인구_수',
              '상주인구':'총_상주인구_수',
              '상권변화지표':'상권_변화_지표',
              '배후지 상주':'배후지_총_상주인구_수',
              '자치구 매핑':'자치구_코드_명',
              '자치구 상주':'자치구_총_상주인구_수',
              '행정동 상주':'행정동_총_상주인구_수'}
join_rate = pd.Series({name: dataset[col].notna().mean()*100 for name, col in check_cols.items()})
join_rate.round(1)

In [ ]:
join_rate.sort_values().plot.barh(figsize=(8,4))
plt.title('데이터별 결합률(%)')
plt.xlim(0,100)
plt.show()

In [ ]:
# 분기별 행 수와 점포 데이터 유무
pd.crosstab(dataset['기준_년분기_코드'], dataset['점포_데이터_있음'])

- 점포 약 1/3 : **점포 파일이 2025년만** 있어서 2023·2024년 행은 비어 있음
  → 점포-상권 2023·2024년 파일을 추가하면 폐업위기(M1) 학습 기간이 12개 분기로 늘어남
- 배후지 상주 : 골목상권만 제공 → 정상

# 8. 저장
- `dataset_상권_업종_분기.csv` : 통합 데이터셋
- `label_source.csv` : 점포 파일 전체(100개 업종) — 다음 분기 폐업률(M1 정답)을 만들 때 사용

In [ ]:
os.makedirs(DATA_DIR + 'processed', exist_ok=True)
store.to_csv(DATA_DIR + 'processed/label_source.csv', index=False, encoding='utf-8-sig')
dataset.to_csv(DATA_DIR + 'processed/dataset_상권_업종_분기.csv', index=False, encoding='utf-8-sig')
print('저장 완료 :', dataset.shape)
dataset.head(3)